<a href="https://colab.research.google.com/github/shivaprabha22/space-debris-predictor/blob/main/orbital_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install skyfield

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 371.1/371.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.1/218.1 kB 9.7 MB/s eta 0:00:00


In [4]:
import requests
import time
from datetime import datetime, timedelta, timezone
from skyfield.api import load
from google.colab import userdata

# Securely retrieve credentials from Colab Secrets
BOT_TOKEN = userdata.get('TELEGRAM_BOT_TOKEN')
CHAT_ID = userdata.get('TELEGRAM_CHAT_ID')

# Infinite execution engine
while True:
    print("\n--- INITIATING ORBITAL SCAN ---")

    # The try/except block prevents network errors from killing the 24/7 loop
    try:
        ts = load.timescale()
        now = datetime.now(timezone.utc)

        print("Downloading space data...")
        stations = load.tle_file('https://celestrak.org/NORAD/elements/gp.php?GROUP=stations&FORMAT=tle', filename='stations.txt')
        iss = {sat.name: sat for sat in stations}['ISS (ZARYA)']

        debris_pieces = load.tle_file('https://celestrak.org/NORAD/elements/gp.php?GROUP=cosmos-2251-debris&FORMAT=tle', filename='debris.txt')

        times_array = ts.from_datetimes([now + timedelta(minutes=m) for m in range(1440)])
        print(f"Scanning ALL {len(debris_pieces)} pieces of Cosmos debris against the ISS. Please wait...")

        absolute_closest_distance = 9999999.0
        absolute_closest_time = ""
        threat_name = ""

        # The One-to-Many Loop
        for debris in debris_pieces:
            distances = (iss - debris).at(times_array).distance().km
            min_distance = min(distances)

            if min_distance < absolute_closest_distance:
                absolute_closest_distance = min_distance
                threat_name = debris.name
                min_index = list(distances).index(min_distance)
                absolute_closest_time = (now + timedelta(minutes=min_index)).strftime('%Y-%m-%d %H:%M:%S UTC')

        # The Alert System Logic
        print("\n--- 24-HOUR THREAT REPORT ---")
        print(f"Primary Target: International Space Station")
        print(f"Greatest Threat: {threat_name}")
        print(f"Approach Distance: {absolute_closest_distance:,.2f} kilometers")
        print(f"Time of Approach: {absolute_closest_time}")
        print("-" * 38)

        # The Threshold (50km)
        if absolute_closest_distance <= 50.0:
            alert_msg = f"🚨 RED ALERT 🚨\nISS Debris Breach Detected!\nThreat: {threat_name}\nDistance: {absolute_closest_distance:,.2f} km"
            url = f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage"
            response = requests.post(url, data={"chat_id": CHAT_ID, "text": alert_msg})
            print("🚨 RED ALERT TRANSMITTED TO PHONE 🚨")
        else:
            print("✅ STATUS GREEN: Orbit is clear for the next 24 hours.")

    except Exception as e:
        print(f"❌ ENGINE ERROR: Network or data failure. {e}")

    print("Scan complete. Engine sleeping for 10 minutes...")
    time.sleep(600)


--- INITIATING ORBITAL SCAN ---


[#################################] 100% stations.txt
[#################################] 100% debris.txt


Scanning ALL 584 pieces of Cosmos debris against the ISS. Please wait...

--- 24-HOUR THREAT REPORT ---
Primary Target: International Space Station
Greatest Threat: COSMOS 2251 DEB
Approach Distance: 81.73 kilometers
Time of Approach: 2026-09-10 23:44:38 UTC
--------------------------------------
✅ STATUS GREEN: Orbit is clear for the next 24 hours.
Scan complete. Engine sleeping for 10 minutes...


KeyboardInterrupt: 